# Smart Fraud Detection Pipeline
### Performance Optimization

This notebook evaluates the performance characteristics of the fraud detection pipeline and demonstrates Spark optimization concepts.

The objective is not to apply optimization techniques unnecessarily,but to understand how Spark executes queries and identify areas where performance improvements may be useful for larger datasets.

#### Optimization Areas

- Query execution plans
- Catalyst Optimizer
- Predicate filtering
- Data scanning
- Delta Lake table characteristics
- Caching considerations
- Partitioning considerations

In [0]:
from pyspark.sql import functions as F

df_transactions = spark.table(
    "fraud_detection.silver.enriched_transactions"
)

df_fraud = spark.table(
    "fraud_detection.gold.fraud_transactions"
)

In [0]:
df_fraud.filter(
    F.col("fraud_status") == "fraud"
).explain(True)

== Parsed Logical Plan ==
'Filter '`==`('fraud_status, fraud)
+- 'UnresolvedRelation [fraud_detection, gold, fraud_transactions], [], false

== Analyzed Logical Plan ==
account_id: string, txn_id: string, txn_date: date, txn_type: string, amount: double, merchant: string, city: string, is_international: boolean, _source_file: string, _ingested_at: timestamp, amount_missing: boolean, amount_valid: boolean, customer_name: string, account_type: string, opening_date: date, branch: string, kyc_status: string, credit_limit: double, account_exists: boolean, fraud_type: string, flagged_date: date, fraud_status: string, fraud_flag: int, fraud_amount: double
Filter (fraud_status#11316 = fraud)
+- SubqueryAlias fraud_detection.gold.fraud_transactions
   +- Relation fraud_detection.gold.fraud_transactions[account_id#11295,txn_id#11296,txn_date#11297,txn_type#11298,amount#11299,merchant#11300,city#11301,is_international#11302,_source_file#11303,_ingested_at#11304,amount_missing#11305,amount_valid#1

#### Demonstration of  Predicate Filtering

In [0]:
fraud_only = (
    df_fraud
    .filter(F.col("fraud_status") == "fraud")
)

display(fraud_only)

account_id,txn_id,txn_date,txn_type,amount,merchant,city,is_international,_source_file,_ingested_at,amount_missing,amount_valid,customer_name,account_type,opening_date,branch,kyc_status,credit_limit,account_exists,fraud_type,flagged_date,fraud_status,fraud_flag,fraud_amount
ACC-00056,TXN-000001,2026-01-09,DEBIT,915931.22,UNKNOWN,Singapore,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:45:45.409Z,false,true,null,null,null,null,null,null,false,ACCOUNT_TAKEOVER,2026-02-20,fraud,1,915931.22
ACC-00029,TXN-000003,2026-02-03,CREDIT,15277.03,Swiggy,New York,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:45:45.409Z,false,true,Tanuja Nair,SALARY,2024-05-02,Pune_FC,VERIFIED,100000.0,true,MONEY_LAUNDERING,2026-01-13,fraud,1,15277.03
ACC-00029,TXN-000005,2026-02-23,DEBIT,919.78,Flipkart,Chennai,false,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:45:45.409Z,false,true,Tanuja Nair,SALARY,2024-05-02,Pune_FC,VERIFIED,100000.0,true,MONEY_LAUNDERING,2026-01-13,fraud,1,919.78
ACC-00021,TXN-000025,2026-03-04,PAYMENT,13669.83,BigBasket,Lagos,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:45:45.409Z,false,true,Sanjay Bhat,NRI,2020-04-06,Chennai_T_Nagar,PENDING,1000000.0,true,CARD_CLONING,2026-01-14,fraud,1,13669.83
ACC-00038,TXN-000028,2026-01-02,DEBIT,19786.75,ATM,Chennai,false,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:45:45.409Z,false,true,Sushma Thakur,CURRENT,2022-07-16,Bangalore_MG,VERIFIED,1000000.0,true,CARD_CLONING,2026-01-13,fraud,1,19786.75
ACC-00038,TXN-000041,2026-02-19,WITHDRAWAL,798127.09,ATM,London,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:45:45.409Z,false,true,Sushma Thakur,CURRENT,2022-07-16,Bangalore_MG,VERIFIED,1000000.0,true,CARD_CLONING,2026-01-13,fraud,1,798127.09
ACC-00009,TXN-000045,2026-03-15,CREDIT,15818.93,BookMyShow,Lagos,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:45:45.409Z,false,true,Venkat Naidu,SAVINGS,2017-11-05,Chennai_T_Nagar,VERIFIED,500000.0,true,IDENTITY_THEFT,2026-01-18,fraud,1,15818.93
ACC-00031,TXN-000050,2026-02-27,TRANSFER,411.87,PhonePe,Lagos,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:45:45.409Z,false,true,Akash Dhawan,CURRENT,2023-02-28,Kolkata_Park,VERIFIED,100000.0,true,PHISHING,2026-03-08,fraud,1,411.87
ACC-00009,TXN-000065,2026-01-11,PAYMENT,2883.26,Airtel,Chennai,false,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:45:45.409Z,false,true,Venkat Naidu,SAVINGS,2017-11-05,Chennai_T_Nagar,VERIFIED,500000.0,true,IDENTITY_THEFT,2026-01-18,fraud,1,2883.26
ACC-00029,TXN-000068,2026-03-16,PAYMENT,20747.52,Airtel,Tokyo,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:45:45.409Z,false,true,Tanuja Nair,SALARY,2024-05-02,Pune_FC,VERIFIED,100000.0,true,MONEY_LAUNDERING,2026-01-13,fraud,1,20747.52


This reduces unnecessary downstream processing.

#### Check table statistics

In [0]:
spark.sql("""
DESCRIBE DETAIL fraud_detection.gold.fraud_transactions
""").display()

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,303ffd54-804c-4a29-9072-4386da24343f,fraud_detection.gold.fraud_transactions,null,,2026-08-08T14:15:10.220Z,2026-08-08T14:46:49.000Z,List(),List(),1,11308,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


This tells us about how Delta table is stored.

### Performance Considerations

Although partitioning can improve performance for large datasets, here we have avoided unnecessary partitioning because the current dataset is small. Excessive partitioning can create many small files and increase overhead.